# Step 0 — Build the raw salary dataset

Source: [LinkedIn Job Postings dataset on Kaggle](https://www.kaggle.com/datasets/arshkon/linkedin-job-postings)



In [1]:
!pip install -q kaggle datasets huggingface_hub pandas
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [2]:
import os
import random
import pandas as pd
from huggingface_hub import login
from job_items import JobItem
from google.colab import userdata

login(userdata.get('HF_TOKEN'), add_to_git_credential=True)

## Download the dataset from Kaggle

In [3]:
!kaggle datasets download -d arshkon/linkedin-job-postings -p ./data --unzip

Dataset URL: https://www.kaggle.com/datasets/arshkon/linkedin-job-postings
License(s): CC-BY-SA-4.0
100% 159M/159M [00:04<00:00, 37.7MB/s]



In [4]:
CSV_PATH = "./data/postings.csv"
df = pd.read_csv(CSV_PATH)
print(f"Raw rows: {len(df):,}")
print(df.columns.tolist())

Raw rows: 123,849
['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']


## Filter to rows with usable salary + description

In [5]:
df = df.dropna(subset=["description"])
df = df[df["description"].str.len() > 100]  # drop near-empty descriptions

# Prefer normalized_salary if present; else derive from min/max
if "normalized_salary" in df.columns:
    df["salary_annual"] = df["normalized_salary"]
else:
    df["salary_annual"] = df[["min_salary", "max_salary"]].mean(axis=1)

# Annualize hourly pay if pay_period is present
if "pay_period" in df.columns:
    hourly_mask = df["pay_period"].str.upper().eq("HOURLY")
    df.loc[hourly_mask, "salary_annual"] = df.loc[hourly_mask, "salary_annual"] * 2080  # 40hr/wk * 52wk

df = df.dropna(subset=["salary_annual"])

# Sanity-bound: drop obvious junk (typos, stipends, placeholder values)
df = df[(df["salary_annual"] >= 15000) & (df["salary_annual"] <= 800000)]

print(f"Rows with usable salary + description: {len(df):,}")

Rows with usable salary + description: 20,858


In [6]:
TECH_KEYWORDS = [
    "software", "developer", "engineer", "data scientist", "machine learning",
    "backend", "frontend", "full stack", "devops", "sre", "data engineer",
    "ml engineer", "ai engineer", "programmer",
]
mask = df["title"].str.lower().str.contains("|".join(TECH_KEYWORDS), na=False)
df = df[mask]
print(f"Rows after scoping to tech roles: {len(df):,}")

Rows after scoping to tech roles: 3,107


## Cap dataset size

In [7]:
MAX_ITEMS = 40000
if len(df) > MAX_ITEMS:
    df = df.sample(n=MAX_ITEMS, random_state=42)
print(f"Final dataset size: {len(df):,}")

Final dataset size: 3,107


## Convert rows to JobItem objects

In [8]:
def row_to_jobitem(row, idx) -> JobItem:
    title = str(row.get("title", ""))[:150]
    company = str(row.get("company_name", ""))
    location = str(row.get("location", ""))
    description = str(row["description"])[:3000]

    summary = f"Job Title: {title}\nCompany: {company}\nLocation: {location}\n\n{description}"
    category = "Technology"

    return JobItem(
        title=title,
        category=category,
        salary=float(row["salary_annual"]),
        full=summary,
        summary=summary,
        id=idx,
    )

items = [row_to_jobitem(row, i) for i, row in df.reset_index(drop=True).iterrows()]
print(f"Built {len(items):,} JobItems")
print(items[0])

Built 3,107 JobItems
title='Building Engineer' category='Technology' salary=105000.0 full='Job Title: Building Engineer\nCompany: nan\nLocation: San Francisco, CA\n\nSummary: Due to the pending retirement of our building engineer, we are seeking a Building Engineer (BE). The BE is a salaried, overtime-exempt professional with direct responsibility for the physical plant of our historic clubhouse. This hands-on position involves light maintenance tasks, operation of building systems, selection and oversight of outside contractors, and administration of building maintenance records. \nFounded in 1852, the Pacific-Union Club is one of the oldest and most exclusive clubs in the world and is known the world over for its excellent facilities and gracious staff. Our 1910 clubhouse is a National Historic Landmark and a California Designated Landmark. The Club provides dining services, a library, athletic facilities, and overnight accommodation.  Qualifications:· Professional training certifica

## Train / val / test split

In [9]:
random.seed(42)
random.shuffle(items)

n = len(items)
n_test = min(2000, int(n * 0.05))
n_val = min(2000, int(n * 0.05))

test = items[:n_test]
val = items[n_test:n_test + n_val]
train = items[n_test + n_val:]

print(f"train={len(train):,} val={len(val):,} test={len(test):,}")

train=2,797 val=155 test=155


In [11]:
username = "toughfigure"
JobItem.push_to_hub(f"{username}/jobs_full", train, val, test)
print("Pushed to hub. Ready for the Day 2 prep notebook.")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  12%|#1        |  989kB / 8.45MB            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  503kB /  503kB            

Pushed to hub. Ready for the Day 2 prep notebook.
